In [20]:
!pip install pymupdf groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 55.2 MB/s eta 0:00:00


In [27]:
import pymupdf
import json
from groq import Groq

In [29]:
from google.colab import files

uploaded = files.upload()

Saving ACM_Vectorless_RAG.pdf to ACM_Vectorless_RAG.pdf


In [30]:
pdf_name = list(uploaded.keys())[0]
pdf = pymupdf.open(pdf_name)
print("Number of pages:", len(pdf))

Number of pages: 16


In [31]:
toc = pdf.get_toc()

print("Number of headings:", len(toc))
for entry in toc[:10]:
    print(entry)

Number of headings: 86
[1, '1 Artificial Intelligence', 4]
[2, '1.1 Scope and Definitions', 4]
[2, '1.2 Types of AI by Capability', 4]
[3, '1.2.1 Narrow AI', 4]
[3, '1.2.2 Artificial General Intelligence', 4]
[2, '1.3 Brief History', 4]
[3, '1.3.1 Early Symbolic Era', 5]
[3, '1.3.2 Statistical Era', 5]
[3, '1.3.3 Deep Learning Era', 5]
[1, '2 Machine Learning', 5]


In [32]:
full_text = ""
page_start = []

for page in pdf:
    page_start.append(len(full_text))
    full_text += " ".join(page.get_text().split()) + " "

print(full_text[:1000])

Artificial Intelligence: A Hierarchical Overview From the Field as a Whole to Its Sub-domains and Techniques This document organises the field of Artificial Intelligence (AI) as a tree. Section headings are the top-level branches, subsections are their children, and subsubsections are the specific techniques or concepts beneath them. Each section begins with a line stating its parent and children so that its place in the hierarchy is explicit. Contents 1 Artificial Intelligence 4 1.1 Scope and Definitions . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4 1.2 Types of AI by Capability . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4 1.2.1 Narrow AI . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4 1.2.2 Artificial General Intelligence . . . . . . . . . . . . . . . . . . . . . . . . . 4 1.3 Brief History . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4 1.3.1 Early Symbolic Era . . . . . . . . . . . . . . 

In [33]:
positions = []
prev = 0

for level, title, page in toc:
    start = max(prev, page_start[page - 1])
    clean_title = " ".join(title.split())
    pos = full_text.find(clean_title, start)
    if pos == -1:
        pos = start
    positions.append(pos)
    prev = pos + 1

print(positions[:10])

[7711, 8288, 8883, 9000, 9254, 9542, 9661, 9993, 10262, 10541]


In [34]:
root = {"id": "0", "title": "Document", "text": "", "children": []}
last_at_level = {0: root}

for i in range(len(toc)):
    level, title, page = toc[i]

    end = len(full_text)
    if i + 1 < len(toc):
        end = positions[i + 1]

    node = {
        "id": "",
        "title": " ".join(title.split()),
        "page": page,
        "text": full_text[positions[i]:end],
        "children": []
    }

    parent = last_at_level[level - 1]
    node["id"] = parent["id"] + "." + str(len(parent["children"]) + 1)
    parent["children"].append(node)
    last_at_level[level] = node

In [35]:
for a in root["children"]:
    print(a["id"], a["title"])
    for b in a["children"]:
        print("    ", b["id"], b["title"])
        for c in b["children"]:
            print("        ", c["id"], c["title"])

0.1 1 Artificial Intelligence
     0.1.1 1.1 Scope and Definitions
     0.1.2 1.2 Types of AI by Capability
         0.1.2.1 1.2.1 Narrow AI
         0.1.2.2 1.2.2 Artificial General Intelligence
     0.1.3 1.3 Brief History
         0.1.3.1 1.3.1 Early Symbolic Era
         0.1.3.2 1.3.2 Statistical Era
         0.1.3.3 1.3.3 Deep Learning Era
0.2 2 Machine Learning
     0.2.1 2.1 Supervised Learning
         0.2.1.1 2.1.1 Regression
         0.2.1.2 2.1.2 Classification
     0.2.2 2.2 Unsupervised Learning
         0.2.2.1 2.2.1 Clustering
         0.2.2.2 2.2.2 Dimensionality Reduction
         0.2.2.3 2.2.3 Anomaly Detection
     0.2.3 2.3 Semi-supervised and Self-supervised Learning
     0.2.4 2.4 Model Evaluation
         0.2.4.1 2.4.1 Metrics
         0.2.4.2 2.4.2 Overfitting and Regularisation
0.3 3 Deep Learning
     0.3.1 3.1 Neural Network Foundations
         0.3.1.1 3.1.1 Perceptron and Multilayer Networks
         0.3.1.2 3.1.2 Activation Functions
         0.3.1.3 3.1.3

In [37]:
with open("tree.json", "w") as f:
    json.dump(root, f, indent=2)

print("Saved tree.json")

Saved tree.json


In [38]:
from google.colab import userdata

api_key = userdata.get('grok')
client = Groq(api_key=api_key)

In [39]:
def choose_sections(question, nodes):
    menu = ""
    for n in nodes:
        menu += n["id"] + " | " + n["title"] + " | " + n["text"][:200] + "\n"

    prompt = f"""
Question: {question}

Below is a list of document sections (id | title | preview).
Pick the sections most likely to contain the answer.
Reply with ONLY the ids separated by commas, like: 0.1,0.3
If none are relevant, reply with: none

{menu}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}]
    )

    reply = response.choices[0].message.content
    ids = reply.replace(" ", "").replace("\n", "").split(",")

    chosen = []
    for n in nodes:
        if n["id"] in ids:
            chosen.append(n)

    return chosen

In [40]:
question = "What is the difference between LSTM and GRU?"

In [41]:
level1 = choose_sections(question, root["children"])

for n in level1:
    print(n["id"], n["title"])

0.3 3 Deep Learning


In [42]:
options = []
for n in level1:
    options += n["children"]

level2 = choose_sections(question, options)

for n in level2:
    print(n["id"], n["title"])

0.3.3 3.3 Recurrent Neural Networks


In [43]:
options = []
for n in level2:
    options += n["children"]

level3 = choose_sections(question, options)

for n in level3:
    print(n["id"], n["title"])

0.3.3.2 3.3.2 LSTM
0.3.3.3 3.3.3 GRU


In [44]:
relevant_nodes = level1 + level2 + level3

context = ""
for n in relevant_nodes:
    context += "[" + n["id"] + "] " + n["title"] + "\n" + n["text"] + "\n\n"

print(context[:1500])

[0.3] 3 Deep Learning
3 Deep Learning Parent: Machine Learning. Children: Neural Network Foundations, Convolutional Neural Networks, Recurrent Neural Networks, Transformers, Generative Models. Deep Learning is the subset of Machine Learning that uses artificial neural networks with many layers. It learns hierarchical representations directly from raw data, which removes much of the manual feature engineering needed in classical ML. 

[0.3.3] 3.3 Recurrent Neural Networks
3.3 Recurrent Neural Networks Parent: Deep Learning. Children: Vanilla RNN, LSTM, GRU. Recurrent Neural Networks process sequences one element at a time and carry a hidden state forward. 

[0.3.3.2] 3.3.2 LSTM
3.3.2 LSTM Parent: Recurrent Neural Networks. Long Short-Term Memory networks, introduced by Hochreiter and Schmidhuber in 1997, use input, forget, and output gates and a cell state to preserve information over long sequences. 

[0.3.3.3] 3.3.3 GRU
3.3.3 GRU Parent: Recurrent Neural Networks. The Gated Recurrent 

In [45]:
prompt = f"""
Answer the question using only the context below.
Mention the section ids you used.

Context:
{context}

Question:
{question}
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)

answer = response.choices[0].message.content
print(answer)

LSTM and GRU are both gated variants of recurrent neural networks, but they differ in their gate structure and the use of a separate memory cell:

* **LSTM** – uses three gates (input, forget, and output) together with an explicit cell state that carries long‑term information.  
  (Section **[0.3.3.2]**)

* **GRU** – employs only two gates (update and reset) and does not maintain a separate cell state; its hidden state serves as both hidden and memory representation.  
  (Section **[0.3.3.3]**)


# UI for LinkedIn

In [46]:
def ask_rag(question):
    level1 = choose_sections(question, root["children"])

    options = []
    for n in level1:
        options += n["children"]
    level2 = choose_sections(question, options)

    options = []
    for n in level2:
        options += n["children"]
    level3 = choose_sections(question, options)

    relevant_nodes = level1 + level2 + level3

    context = ""
    trace = ""
    for n in relevant_nodes:
        context += "[" + n["id"] + "] " + n["title"] + "\n" + n["text"] + "\n\n"
        trace += n["id"] + "  " + n["title"] + "  (page " + str(n["page"]) + ")\n"

    prompt = f"""
    Answer the question using only the context below.
    Mention the section ids you used.

    Context:
    {context}

    Question:
    {question}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content, trace

In [47]:
import gradio as gr

demo = gr.Interface(
    fn=ask_rag,
    inputs=gr.Textbox(
        label="Ask a question",
        placeholder="Ask something about the document..."
    ),
    outputs=[
        gr.Textbox(label="Vectorless RAG Assistant", lines=8),
        gr.Textbox(label="Sections used (tree path)", lines=6)
    ],
    title="Vectorless RAG Assistant",
    description="No embeddings. No vector database. The AI walks the document tree."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a85d5aad368968e39c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
